# CS410 Project -- Phase 1
Cindy Nguyen

Prof. Michael Wilson

TA. Amber Shore

CS410 TOP: Data With Python

1/17/2026

## Project Overview
VocaDB is a fanmade wiki that contains all the information about VOCALOID, primarily songs and setlists for concerts around the world.

VOCALOID and its official concerts, along with special music events are owned and hosted by Crypton Future Media (CFM).

Although CFM has official concerts, there are also many fanmade events as well around the world, and other companies that host vocal synth concerts around the world.

VocaDB has an API which is where I'm getting my data from. I want to use this project to predict:
1. What songs will likely show up at the next VOCALOID concert in April (MIKU EXPO 2026)?
2. When will those songs play in the setlist? What is their average position?

## Importing VocaDB data from .json files
All of the data pulled from the API is stored in .json format. I created a .py script to pull all of this data from VocaDB, and formatted it a little nicer so it's easier to use/query from.

This is all of the songs from previous concert/event setlists.

In [1]:
import pandas as pd
import json

songs_df = pd.read_json("vocaloid_setlist_songs.json")
songs_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2274 entries, 0 to 2273
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              2274 non-null   str    
 1   english_title      2274 non-null   str    
 2   artist             2274 non-null   str    
 3   length_seconds     2274 non-null   int64  
 4   setlist_frequency  2274 non-null   int64  
 5   avg_setlist_order  2274 non-null   float64
 6   song_type          2274 non-null   str    
 7   pv_services        2274 non-null   str    
 8   times_favorited    2274 non-null   int64  
 9   rating             2274 non-null   int64  
dtypes: float64(1), int64(4), str(5)
memory usage: 177.8 KB


This looks pretty great! There's no `NaN` values at all, pretty much every field is populated so this cleaning step isn't necessary.

## Data Cleansing
Even if there are no `NaN` values, we still have to clean the data. And given that this is numerical data, cleaning it is sort of a different process.

### Computing the average setlist order (avg_setlist_order)
One of the data cleansing steps actually has to be done outside of Jupyter notebook since it has to do with how I retrieved my data and computed the average setlist order when creating the dataset itself.

VOCALOID is a big community of fans, full of very talented producers, musicians, and also DJs. This doesn't sound terrible until you realize that these special DJ events can go on for a really long time and mashup tons of songs together.

This is how long the setlist is for the MIKU EXPO Digital Stars 2021 Online event...

<img src="bad_data.png"></img>

93 songs is a LOT. Fortunately, "MONSTER" was only ever played once at this special event so it's not going to skew the data for other instances of it being played.

However, there were other songs that are much more popular that were in this setlist as well (Melt, Glass Wall, to name a few). These songs would typically show up much earlier in the setlist across multiple regular concerts, but are played much later at DJ events to build up excitement for the audience. 

This is a huge problem, since this will compute the average order to be much higher than it's supposed to be.

It's hard to demonstrate here, but what I'm going to do is exclude any setlists with > 50 songs from my dataset by modifying my Python script.

__**I can do this relatively safely for a few reasons**__:
1. From my own personal knowledge, most concerts average to around ~30 songs total
    - There are a couple exceptions:
       - Kikuoland (49)
       - Project SEKAI COLORFUL LIVE 4th - Unison (48)
       - VVV MUSIC LIVE concert series (~56)
2. Choosing the number "50" will cover the two main exceptions above, but exclude any super long special events (i.e. DJ sets) which are the MOST likely to skew the value of the average order negatively
3. It's safe to not include any of the VVV MUSIC LIVE concerts (or care at all about VVV MUSIC LIVE)
   - These concerts use voice/character models that would never show up at a MIKU EXPO concert due to licensing reasons UNLESS a major breakthrough happens where a collaboration is viable
      - Nowadays, this is incredibly rare, and in modern times has only happened ONCE with 32ki's song "Mesmerizer" where Kasane Teto made a guest appearance for the JAPAN LIVE TOUR 2025 ~ BLOOMING ~ (and this isn't even a main concert event!)
   - Even if any of the performers had a chance of appearing at MIKU EXPO (e.g. GUMI since she is partially licensed under CFM still), the songs from VVV MUSIC LIVE would never be performed at MIKU EXPO due to licensing reasons

### Before/After Results of Removing and Recalculating

#### Here is the dataset *before* removing all setlists longer than 50 tracks, along with the average positions computed including those long setlists.

In [7]:
songs_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2274 entries, 0 to 2273
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              2274 non-null   str    
 1   english_title      2274 non-null   str    
 2   artist             2274 non-null   str    
 3   length_seconds     2274 non-null   int64  
 4   setlist_frequency  2274 non-null   int64  
 5   avg_setlist_order  2274 non-null   float64
 6   song_type          2274 non-null   str    
 7   pv_services        2274 non-null   str    
 8   times_favorited    2274 non-null   int64  
 9   rating             2274 non-null   int64  
dtypes: float64(1), int64(4), str(5)
memory usage: 177.8 KB


In [ ]:
# Setting this to 30 so we can actually see the avg_setlist_order be > 1

mask = songs_df["setlist_frequency"] > 30

frequent_songs = songs_df[mask]

frequent_songs

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
64,千本桜,A Thousand Cherry Blossoms,黒うさP feat. 初音ミク,246,37,14.270270,Original,"NicoNicoDouga, Youtube",388,1858
73,ルカルカ★ナイトフィーバー,Luka Luka★Night Fever,samfree feat. 巡音ルカ,239,39,16.307692,Original,"NicoNicoDouga, Youtube",261,1329
83,メルト,Melt,ryo feat. 初音ミク,256,41,24.292683,Original,"NicoNicoDouga, Youtube, SoundCloud",297,1522
149,リモコン,Remote Control,"じーざすP, WONDERFUL★OPPORTUNITY! feat. 鏡音リン, 鏡音レン",312,31,14.419355,Original,"NicoNicoDouga, Youtube",206,1086
152,ワールドイズマイン,World is Mine,"ryo, supercell feat. 初音ミク",255,62,13.500000,Original,"NicoNicoDouga, Youtube",303,1609
188,???,None,,0,75,19.946667,Other,Nothing,2,12
268,Tell Your World,None,kz feat. 初音ミク,257,66,19.045455,Remix,Youtube,218,1226


#### And here is the new dataset *after* removing setlists with more than 50 songs and recalculating the average positions.

In [27]:
clean_songs_df = pd.read_json("CLEAN_vocaloid_setlist_songs.json")

clean_songs_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2227 entries, 0 to 2226
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              2227 non-null   str    
 1   english_title      2227 non-null   str    
 2   artist             2227 non-null   str    
 3   length_seconds     2227 non-null   int64  
 4   setlist_frequency  2227 non-null   int64  
 5   avg_setlist_order  2227 non-null   float64
 6   song_type          2227 non-null   str    
 7   pv_services        2227 non-null   str    
 8   times_favorited    2227 non-null   int64  
 9   rating             2227 non-null   int64  
dtypes: float64(1), int64(4), str(5)
memory usage: 174.1 KB


In [ ]:
# Setting this to 30 so we can actually see the avg_setlist_order be > 1

mask = clean_songs_df["setlist_frequency"] > 30

clean_frequent_songs = clean_songs_df[mask]

clean_frequent_songs

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
64,千本桜,A Thousand Cherry Blossoms,黒うさP feat. 初音ミク,246,36,12.361111,Original,"NicoNicoDouga, Youtube",388,1858
73,ルカルカ★ナイトフィーバー,Luka Luka★Night Fever,samfree feat. 巡音ルカ,239,39,16.307692,Original,"NicoNicoDouga, Youtube",261,1329
83,メルト,Melt,ryo feat. 初音ミク,256,41,24.292683,Original,"NicoNicoDouga, Youtube, SoundCloud",297,1522
149,リモコン,Remote Control,"じーざすP, WONDERFUL★OPPORTUNITY! feat. 鏡音リン, 鏡音レン",312,31,14.419355,Original,"NicoNicoDouga, Youtube",206,1086
152,ワールドイズマイン,World is Mine,"ryo, supercell feat. 初音ミク",255,62,13.500000,Original,"NicoNicoDouga, Youtube",303,1609
188,???,None,,0,65,13.784615,Other,Nothing,2,12
268,Tell Your World,None,kz feat. 初音ミク,257,65,18.169231,Remix,Youtube,218,1226


#### Results
If you're like me and couldn't tell the difference at first, here's a specific example using "Senbonzakura":

In [14]:
frequent_songs.head(1)

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
64,千本桜,A Thousand Cherry Blossoms,黒うさP feat. 初音ミク,246,37,14.27027,Original,"NicoNicoDouga, Youtube",388,1858


In [25]:
clean_frequent_songs.head(1)

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
64,千本桜,A Thousand Cherry Blossoms,黒うさP feat. 初音ミク,246,36,12.361111,Original,"NicoNicoDouga, Youtube",388,1858


The average setlist position changed! While that's only 2 spots earlier in the setlist for this particular song, that's still easily about 10 minutes of your life waiting for a fan favorite title to play.

2 spots isn't that big of a deal, but for other songs that are even more popular and were included in larger DJ setlists, the changes in the average setlist position could've been much more dramatic.

### Songs with 0 seconds in length

So you might've noticed something funky earlier, and it's that there are songs that are 0 seconds in duration.

In [28]:
zero_length_songs = clean_songs_df["length_seconds"] == 0
clean_songs_df[zero_length_songs]

,title,english_title,artist,length_seconds,setlist_frequency,avg_setlist_order,song_type,pv_services,times_favorited,rating
0,STARLIGHT,None,失いP feat. 花隈千冬,0,1,1.0,Cover,Nothing,0,0
5,物語の始まり,Monogatari no Hajimari,失いP feat. 花隈千冬,0,1,6.0,Cover,Nothing,0,0
9,ビーチストーリー,Beach Story,失いP feat. 京町セイカ AI,0,1,10.0,Cover,Nothing,0,0
10,黒いメガネ,Kuroi Megane,失いP feat. 花隈千冬,0,1,11.0,Cover,Nothing,0,0
11,僕の命,Boku no Inochi,失いP feat. 桜乃そら SV,0,1,12.0,Cover,Nothing,0,0
...,...,...,...,...,...,...,...,...,...,...
2147,地球毁灭的那一刻,When the doomsday comes,"KIDE, 动点P feat. 星尘",0,1,5.0,Original,Nothing,2,10
2182,All About That Bass,None,Dharma feat. 京町セイカ (Unknown),0,1,4.0,Cover,Nothing,1,3
2183,Feel Special,None,ぎゅや feat. 京町セイカ (Unknown),0,1,5.0,Cover,Nothing,0,0
2187,天城越え,Walk Over Amagi Pass,ぎゅや feat. 京町セイカ (Unknown),0,1,11.0,Cover,Nothing,0,0


This information is hidden and won't show up on the website, but according to VocaDB, one of these songs is *supposed* to be "FE!N" by Travis Scott.

<img src="fein.png">

As funny as this is, no, CFM has never collaborated with Travis Scott and FE!N is definitely not a Japanese song.

I mentioned it earlier, but VocaDB is a fanmade wiki. And the "fanmade" parts of it really shine through in moments like this.

All humans make mistakes. Some of the mistakes can be incredibly funny. Maybe someone's engaging in internet trolling to make my life more inconvenient. Though sometimes, human error *just* happens. The other song titles seem very legitimate - and they are.

__**Theories as to why `0` second data happens**__:
- A fan is entering in bad data because they think it's funny (it kind of is (it's really inconvenient though))
- Someone entered the data incorrectly but forgot to remove the information from the database itself
- Some of the songs had actual durations but disappeared over time
   - "Monogatari no Hajimari" is supposed to be 4:27 (`267` seconds), but shows up as `0` because the original link to the song doesn't exist anymore
- Artists delete their old works all the time, and it becomes lost media very quickly in the VOCALOID community to the point that there's a subcommunity that's dedicated to finding the origin of lost songs. If no one archives it, there's no trace of it existing other than basic information that might've been input into the database before it was deleted

Whatever the case may be, we don't really care. I care because I'm a fanatic, but for data processing, cleaning, and the sake of this project, we don't want this data! We're going to get rid of it!

### Songs with no PV services